In [ ]:
import cv2
import numpy as np
from mss import mss
from ultralytics import YOLO

def run_live_inference():
    print("Loading YOLO model...")
    # Load your trained model. It will automatically detect and use Apple Silicon (MPS) if available.
    model = YOLO("best.pt")
    
    # Initialize the ultra-fast screen capture
    sct = mss()
    
    # Grab the dimensions of the primary monitor
    # Note: On Retina displays, this might capture a massive resolution (e.g., 2560x1600+).
    # YOLO handles the downscaling automatically, but we will resize the viewing window so it fits on your screen.
    monitor = sct.monitors[1] 
    
    print("Starting screen capture. Click the window and press 'q' to quit.")

    while True:
        # 1. Capture the screen
        screenshot = sct.grab(monitor)
        
        # 2. Convert raw bytes to a numpy array (MSS captures in BGRA format)
        img_bgra = np.array(screenshot)
        
        # 3. Strip the Alpha channel (Transparency) so it is standard BGR for YOLO
        img_bgr = cv2.cvtColor(img_bgra, cv2.COLOR_BGRA2BGR)
        
        # 4. Run YOLO Inference
        # verbose=False stops the terminal from printing stats 60 times a second
        results = model.predict(source=img_bgr, imgsz=1024, conf=0.3, verbose=False)
        
        # 5. Draw the bounding boxes and labels onto the frame
        annotated_frame = results[0].plot()
        
        # 6. Resize the viewing window to 720p so it doesn't take over your entire Mac display
        display_width = 1280
        display_height = int(display_width * (annotated_frame.shape[0] / annotated_frame.shape[1]))
        view_frame = cv2.resize(annotated_frame, (display_width, display_height))
        
        # 7. Show the frame
        cv2.imshow("YOLOv8 Live Inference", view_frame)
        
        # Press 'q' to cleanly exit the loop
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Clean up windows and memory
    cv2.destroyAllWindows()
    sct.close()

if __name__ == "__main__":
    run_live_inference()

In [ ]:
import time
import threading
import cv2
import numpy as np
from mss import mss
from ultralytics import YOLO

try:
    import torch
    _HAS_TORCH = True
except ImportError:
    _HAS_TORCH = False

# ── Performance knobs ────────────────────────────────────────────────────────────
INFERENCE_IMGSZ     =  1024  # YOLO input size
CAPTURE_DOWNSCALE_W = 1024   # downscale working frame to this width (0 = native)
CONF_THRESHOLD      = 0.3
USE_HALF            = False  # fp16 — only on CUDA; keep False on Apple MPS
INFERENCE_EVERY_N   = 1      # run YOLO every N frames (>1 reuses last detections).
                             # Only helps if inference is the bottleneck — check the
                             # timing overlay first.

# ── HUD Configuration ──────────────────────────────────────────────────────────
BAR_H        = 10
LABEL_H      = 16
PCT_H        = 16
PADDING      = 6
GAP          = 10
HUD_X        = 10
HUD_Y        = 10
MAX_CROP_W   = 300
BADGE_R      = 12

BOX_THICKNESS = 2

# ── Deadzones (fractions of frame, painted black before YOLO) ───────────────────
DEADZONES = [
    (0.80, 0.65, 1.00, 1.00),
    (0.00, 0.00, 0.20, 0.12),
]
SHOW_DEADZONE_OUTLINE = True

# ── Per-class colour palette (BGR) ─────────────────────────────────────────────
CLASS_COLORS = {
    "player":       (  0, 220,   0),
    "ally":         (180,  80,   0),
    "ally_minion":  (255, 140,   0),
    "enemy":        (  0,   0, 180),
    "enemy_minion": ( 60, 100, 220),
    "neutral":      (160, 160, 160),
    "tower":        (  0, 200, 255),
}
DEFAULT_COLOR = (200, 200, 200)


def class_color(label: str):
    return CLASS_COLORS.get(label, DEFAULT_COLOR)


def text_on(color):
    b, g, r = color
    lum = 0.299 * r + 0.587 * g + 0.114 * b
    return (0, 0, 0) if lum > 140 else (255, 255, 255)


# ── HUD ordering ─────────────────────────────────────────────────────────────────
# Returns (category_rank, sub_rank). Lower sorts first.
#   0 champions:  player → allies → enemies
#   1 structures: turrets / towers / inhibitors / nexus
#   2 monsters:   epic + jungle (baron, dragon, herald, neutral camps)
#   3 minions:    ally minions → enemy minions → other
#   4 anything else
def hud_category(label: str):
    l = label.lower()

    if "minion" in l:                       # check minions before champions
        if "ally" in l or "friend" in l:  return (3, 0)
        if "enemy" in l or "hostile" in l: return (3, 1)
        return (3, 2)

    if l == "player" or "self" in l:        return (0, 0)
    if "ally" in l or "friend" in l:        return (0, 1)
    if "enemy" in l or "hostile" in l:      return (0, 2)

    if any(k in l for k in ("turret", "tower", "inhibitor", "nexus", "structure")):
        return (1, 0)

    if any(k in l for k in ("baron", "dragon", "drake", "herald", "epic",
                            "monster", "camp", "jungle", "buff", "neutral")):
        return (2, 0)

    return (4, 0)


# ── Centroid tracker (persistent IDs) ───────────────────────────────────────────
class CentroidTracker:
    def __init__(self, dist_frac=0.06, max_missing=15):
        self.next_id     = 1
        self.objects     = {}
        self.missing     = {}
        self.dist_frac   = dist_frac
        self.max_missing = max_missing

    def update(self, boxes, frame_w):
        max_dist  = self.dist_frac * max(frame_w, 1)
        centroids = [((x1 + x2) / 2.0, (y1 + y2) / 2.0) for (x1, y1, x2, y2) in boxes]
        assigned  = [None] * len(boxes)

        if not self.objects:
            for i, c in enumerate(centroids):
                assigned[i] = self._register(c)
            return assigned

        object_ids       = list(self.objects.keys())
        object_centroids = list(self.objects.values())

        pairs = []
        for di, c in enumerate(centroids):
            for oi, oc in enumerate(object_centroids):
                d = ((c[0] - oc[0]) ** 2 + (c[1] - oc[1]) ** 2) ** 0.5
                pairs.append((d, di, oi))
        pairs.sort(key=lambda p: p[0])

        used_dets, used_objs = set(), set()
        for d, di, oi in pairs:
            if d > max_dist:
                break
            if di in used_dets or oi in used_objs:
                continue
            oid = object_ids[oi]
            self.objects[oid] = centroids[di]
            self.missing[oid] = 0
            assigned[di] = oid
            used_dets.add(di)
            used_objs.add(oi)

        for di, c in enumerate(centroids):
            if di not in used_dets:
                assigned[di] = self._register(c)

        for oi, oid in enumerate(object_ids):
            if oi not in used_objs:
                self.missing[oid] = self.missing.get(oid, 0) + 1
                if self.missing[oid] > self.max_missing:
                    self.objects.pop(oid, None)
                    self.missing.pop(oid, None)

        return assigned

    def _register(self, centroid):
        oid = self.next_id
        self.next_id += 1
        self.objects[oid] = centroid
        self.missing[oid] = 0
        return oid


# ── Health bar estimation ───────────────────────────────────────────────────────
BLACK_THRESHOLD      = 40
DARK_RATIO_THRESHOLD = 0.55


def estimate_health_pct(crop: np.ndarray) -> float:
    if crop is None or crop.size == 0:
        return 0.0
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, w = gray.shape
    col_dark_ratio = (gray < BLACK_THRESHOLD).mean(axis=0)
    is_dark = col_dark_ratio >= DARK_RATIO_THRESHOLD
    lit_cols = np.where(~is_dark)[0]
    if len(lit_cols) == 0:
        return 0.0
    return min(1.0, int(lit_cols[-1]) / max(w - 1, 1))


def health_color(pct: float):
    r = int(255 * (1.0 - pct))
    g = int(255 * pct)
    return (0, g, r)


# ── Deadzone masking ─────────────────────────────────────────────────────────────

def apply_deadzones(frame):
    h, w = frame.shape[:2]
    for (fx1, fy1, fx2, fy2) in DEADZONES:
        x1, y1 = int(fx1 * w), int(fy1 * h)
        x2, y2 = int(fx2 * w), int(fy2 * h)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 0), -1)
        if SHOW_DEADZONE_OUTLINE:
            cv2.rectangle(frame, (x1, y1), (x2, y2), (60, 60, 60), 1)
            cv2.putText(frame, "DEADZONE", (x1 + 4, y1 + 16),
                        cv2.FONT_HERSHEY_PLAIN, 1.0, (90, 90, 90), 1, cv2.LINE_AA)
    return frame


# ── Drawing helpers ─────────────────────────────────────────────────────────────

def draw_id_badge(frame, cx, cy, obj_id, color):
    cv2.circle(frame, (cx, cy), BADGE_R, color, -1)
    cv2.circle(frame, (cx, cy), BADGE_R, (255, 255, 255), 1)
    txt = str(obj_id)
    (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
    cv2.putText(frame, txt, (cx - tw // 2, cy + th // 2),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, text_on(color), 1, cv2.LINE_AA)


def draw_perf(frame, fps, timings, device):
    """Top-right: FPS + per-stage timing breakdown so you can see the bottleneck."""
    lines = [
        f"{fps:4.1f} FPS",
        f"dev:  {device}",
        f"cap:  {timings.get('cap', 0)*1000:4.0f} ms",
        f"inf:  {timings.get('inf', 0)*1000:4.0f} ms",
        f"draw: {timings.get('draw', 0)*1000:4.0f} ms",
    ]
    fw = frame.shape[1]
    y  = 24
    for i, line in enumerate(lines):
        (tw, th), _ = cv2.getTextSize(line, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        x = fw - tw - 12
        cv2.rectangle(frame, (x - 6, y - th - 6), (fw - 2, y + 6), (0, 0, 0), -1)
        col = (0, 255, 0) if i == 0 and fps >= 25 else \
              (0, 200, 255) if i == 0 and fps >= 15 else \
              (0, 80, 255) if i == 0 else (210, 210, 210)
        cv2.putText(frame, line, (x, y), cv2.FONT_HERSHEY_SIMPLEX, 0.55,
                    col, 1, cv2.LINE_AA)
        y += th + 12
    return frame


def draw_boxes(frame, dets):
    for d in dets:
        x1, y1, x2, y2 = d["box"]
        color = d["color"]
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, BOX_THICKNESS)
        draw_id_badge(frame, x1 + BADGE_R, y1 - BADGE_R, d["id"], color)
    return frame


def safe_place(frame, patch, top, left):
    fh, fw = frame.shape[:2]
    ph, pw = patch.shape[:2]
    top  = max(0, top)
    left = max(0, left)
    if top >= fh or left >= fw:
        return 0, 0
    h = min(ph, fh - top)
    w = min(pw, fw - left)
    if h <= 0 or w <= 0:
        return 0, 0
    frame[top:top + h, left:left + w] = patch[:h, :w]
    return h, w


def draw_hud(frame, original_frame, dets):
    if not dets:
        return frame

    img_h, img_w = original_frame.shape[:2]

    cards = []
    for d in dets:
        x1, y1, x2, y2 = d["box"]
        cy1, cy2 = max(0, y1), min(img_h, y2)
        cx1, cx2 = max(0, x1), min(img_w, x2)
        crop = original_frame[cy1:cy2, cx1:cx2].copy()

        if crop.shape[1] > MAX_CROP_W and crop.shape[1] > 0:
            scale = MAX_CROP_W / crop.shape[1]
            new_h = max(1, int(crop.shape[0] * scale))
            crop  = cv2.resize(crop, (MAX_CROP_W, new_h), interpolation=cv2.INTER_AREA)

        health_pct = estimate_health_pct(crop)
        cat = hud_category(d["label"])
        cards.append({
            "crop": crop, "label": d["label"], "health": health_pct,
            "color": d["color"], "id": d["id"], "cat": cat,
        })

    # Order: category first (champions→structures→monsters→minions),
    # then by ID within a category so positions stay stable.
    cards.sort(key=lambda c: (c["cat"][0], c["cat"][1], c["id"]))

    max_crop_w = max(c["crop"].shape[1] for c in cards)
    panel_w    = max_crop_w + PADDING * 2 + BADGE_R
    panel_h    = GAP
    for c in cards:
        panel_h += c["crop"].shape[0] + BAR_H + PCT_H + LABEL_H + PADDING * 3 + GAP

    fh, fw = frame.shape[:2]
    px2 = min(HUD_X + panel_w, fw)
    py2 = min(HUD_Y + panel_h, fh)
    overlay = frame.copy()
    cv2.rectangle(overlay, (HUD_X, HUD_Y), (px2, py2), (20, 20, 20), -1)
    cv2.addWeighted(overlay, 0.55, frame, 0.45, 0, frame)

    cursor_y = HUD_Y + GAP
    for c in cards:
        crop, label, health_pct, color, obj_id = (
            c["crop"], c["label"], c["health"], c["color"], c["id"]
        )
        card_x = HUD_X + PADDING

        placed_h, placed_w = safe_place(frame, crop, cursor_y, card_x)

        cv2.rectangle(frame,
                      (card_x, cursor_y),
                      (card_x + placed_w, cursor_y + placed_h),
                      color, 2)
        draw_id_badge(frame, card_x + BADGE_R, cursor_y + BADGE_R, obj_id, color)
        cursor_y += placed_h + PADDING

        pct_text = f"{int(health_pct * 100)}%"
        cv2.putText(frame, pct_text,
                    (card_x, cursor_y + PCT_H - 4),
                    cv2.FONT_HERSHEY_PLAIN, 1.1,
                    (220, 220, 220), 1, cv2.LINE_AA)
        cursor_y += PCT_H

        bar_end_x = card_x + placed_w
        cv2.rectangle(frame, (card_x, cursor_y), (bar_end_x, cursor_y + BAR_H),
                      (50, 50, 50), -1)
        fill_x2 = card_x + int(placed_w * health_pct)
        cv2.rectangle(frame, (card_x, cursor_y), (fill_x2, cursor_y + BAR_H),
                      health_color(health_pct), -1)
        cursor_y += BAR_H + PADDING

        cv2.putText(frame, label,
                    (card_x, cursor_y + LABEL_H - 4),
                    cv2.FONT_HERSHEY_PLAIN, 1.0,
                    color, 1, cv2.LINE_AA)
        cursor_y += LABEL_H + GAP

    return frame


# ── Threaded screen capture ──────────────────────────────────────────────────────
# Capture runs on its own thread so the ~45ms grab overlaps with inference instead
# of stacking on top of it. The main loop just reads the most recent frame.
class ScreenGrabber(threading.Thread):
    def __init__(self, monitor_index, downscale_w):
        super().__init__(daemon=True)
        self.monitor_index = monitor_index
        self.downscale_w   = downscale_w
        self.latest        = None
        self.lock          = threading.Lock()
        self.running       = True
        self.cap_ms        = 0.0
        self.monitor_info  = None
        self._first        = threading.Event()

    def run(self):
        # mss must be created AND used on the same thread.
        local_sct = mss()
        monitor = local_sct.monitors[self.monitor_index]
        self.monitor_info = monitor
        while self.running:
            t0 = time.perf_counter()
            raw   = np.asarray(local_sct.grab(monitor))
            frame = raw[:, :, :3]
            if self.downscale_w and frame.shape[1] > self.downscale_w:
                scale = self.downscale_w / frame.shape[1]
                new_h = int(frame.shape[0] * scale)
                frame = cv2.resize(frame, (self.downscale_w, new_h),
                                   interpolation=cv2.INTER_AREA)
            else:
                frame = np.ascontiguousarray(frame)
            with self.lock:
                self.latest = frame
                self.cap_ms = (time.perf_counter() - t0) * 1000.0
            self._first.set()
        local_sct.close()

    def wait_first(self, timeout=10):
        return self._first.wait(timeout)

    def get(self):
        with self.lock:
            if self.latest is None:
                return None
            return self.latest.copy()

    def stop(self):
        self.running = False


# ── Device selection ─────────────────────────────────────────────────────────────

def pick_device():
    if not _HAS_TORCH:
        return "cpu"
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


# ── Main loop ───────────────────────────────────────────────────────────────────

def run_live_inference():
    print("Loading YOLO model...")
    model = YOLO("best.pt")

    device = pick_device()
    print(f"PyTorch available: {_HAS_TORCH}")
    if _HAS_TORCH:
        print(f"  MPS available : {torch.backends.mps.is_available()}")
        print(f"  CUDA available: {torch.cuda.is_available()}")
    print(f"Using device: {device.upper()}")
    try:
        model.to(device)
    except Exception as e:
        print(f"  (model.to({device}) failed: {e}; predict will still pass device=)")

    class_names = model.names if hasattr(model, "names") else None
    if isinstance(class_names, dict):
        class_names = [class_names[k] for k in sorted(class_names.keys())]

    print("Warming up model...")
    dummy = np.zeros((INFERENCE_IMGSZ, INFERENCE_IMGSZ, 3), dtype=np.uint8)
    model.predict(source=dummy, imgsz=INFERENCE_IMGSZ, conf=CONF_THRESHOLD,
                  verbose=False, half=USE_HALF, device=device)

    tracker = CentroidTracker(dist_frac=0.06, max_missing=15)

    # List monitors (use a temporary mss just for enumeration)
    tmp = mss()
    print("Available monitors:")
    for i, m in enumerate(tmp.monitors):
        print(f"  [{i}] {m['width']}x{m['height']}  offset ({m['left']}, {m['top']})")
    n_monitors = len(tmp.monitors)
    tmp.close()

    GAME_MONITOR_INDEX = 1
    if GAME_MONITOR_INDEX >= n_monitors:
        print(f"Monitor {GAME_MONITOR_INDEX} not found, falling back to 1.")
        GAME_MONITOR_INDEX = 1

    # Start the background capture thread
    grabber = ScreenGrabber(GAME_MONITOR_INDEX, CAPTURE_DOWNSCALE_W)
    grabber.start()
    print("Waiting for first frame...")
    if not grabber.wait_first(timeout=10):
        print("No frame captured — aborting.")
        grabber.stop()
        return
    mon = grabber.monitor_info
    print(f"Capturing monitor [{GAME_MONITOR_INDEX}]: {mon['width']}x{mon['height']}")
    print("Press 'q' to quit.")

    fps_ema = None
    prev_t  = time.perf_counter()
    timings = {"cap": 0.0, "inf": 0.0, "draw": 0.0}
    frame_idx = 0
    dets = []

    while True:
        # ── Stage 1: fetch latest frame from the capture thread ─────────────────
        frame = grabber.get()
        if frame is None:
            continue
        apply_deadzones(frame)
        t1 = time.perf_counter()

        # ── Stage 2: inference (optionally every N frames) ──────────────────────
        frame_idx += 1
        if frame_idx % INFERENCE_EVERY_N == 0:
            results = model.predict(source=frame, imgsz=INFERENCE_IMGSZ,
                                    conf=CONF_THRESHOLD, verbose=False,
                                    half=USE_HALF, device=device)
            new_dets = []
            boxes = results[0].boxes
            if boxes is not None:
                for box in boxes:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    cls_id = int(box.cls[0])
                    label  = class_names[cls_id] if class_names and cls_id < len(class_names) else f"cls{cls_id}"
                    new_dets.append({"box": (x1, y1, x2, y2), "label": label})
            ids = tracker.update([d["box"] for d in new_dets], frame.shape[1])
            for d, i in zip(new_dets, ids):
                d["id"]    = i
                d["color"] = class_color(d["label"])
            dets = new_dets
        t2 = time.perf_counter()

        # ── Stage 3: draw ────────────────────────────────────────────────────────
        annotated = frame.copy()
        draw_boxes(annotated, dets)
        draw_hud(annotated, frame, dets)
        t3 = time.perf_counter()

        # ── FPS + timing overlay ──────────────────────────────────────────────────
        now = time.perf_counter()
        dt  = now - prev_t
        prev_t = now
        if dt > 0:
            inst = 1.0 / dt
            fps_ema = inst if fps_ema is None else 0.9 * fps_ema + 0.1 * inst

        # Smooth the stage timings. cap is measured inside the capture thread.
        a = 0.9
        timings["cap"]  = grabber.cap_ms / 1000.0
        timings["inf"]  = a * timings["inf"]  + (1 - a) * (t2 - t1)
        timings["draw"] = a * timings["draw"] + (1 - a) * (t3 - t2)

        draw_perf(annotated, fps_ema or 0.0, timings, device)

        cv2.imshow("YOLOv8 Live Inference", annotated)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    grabber.stop()
    grabber.join(timeout=1)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    run_live_inference()

In [ ]:
import time
import threading
import cv2
import numpy as np
from mss import mss
from ultralytics import YOLO

try:
    import torch
    _HAS_TORCH = True
except ImportError:
    _HAS_TORCH = False

# ── Performance knobs ────────────────────────────────────────────────────────
INFERENCE_IMGSZ     = 1440
CAPTURE_DOWNSCALE_W = 0
CONF_THRESHOLD      = 0.5
USE_HALF            = False
INFERENCE_EVERY_N   = 2

# ── Deadzones (fractions of frame) ───────────────────────────────────────────
DEADZONES = [
    (0.80, 0.65, 1.00, 1.00),
    (0.00, 0.00, 0.20, 0.12),
]

# ── How far below the health bar the expanded box extends (pixels) ───────────
# The YOLO box is the health bar. We project a larger box below it to
# represent the champion / minion / structure body under the bar.
BODY_EXTEND_DOWN  = 80   # how many px below the health bar bottom
BODY_WIDEN_EACH   = 10   # extra px added to each side (left & right)

# ── Per-class colour palette (BGR) ───────────────────────────────────────────
CLASS_COLORS = {
    "player":       (  0, 220,   0),
    "ally":         (180,  80,   0),
    "ally_minion":  (255, 140,   0),
    "enemy":        (  0,   0, 220),
    "enemy_minion": ( 60, 100, 220),
    "neutral":      (160, 160, 160),
    "tower":        (  0, 200, 255),
}
DEFAULT_COLOR = (200, 200, 200)

BOX_THICKNESS = 2


def class_color(label: str):
    return CLASS_COLORS.get(label, DEFAULT_COLOR)


# ── Health bar estimation ─────────────────────────────────────────────────────
BLACK_THRESHOLD      = 40
DARK_RATIO_THRESHOLD = 0.55


def estimate_health_pct(crop: np.ndarray) -> float:
    """Scan columns left→right; rightmost non-dark column = health fill edge."""
    if crop is None or crop.size == 0:
        return 0.0
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    _, w = gray.shape
    col_dark = (gray < BLACK_THRESHOLD).mean(axis=0) >= DARK_RATIO_THRESHOLD
    lit = np.where(~col_dark)[0]
    if len(lit) == 0:
        return 0.0
    return min(1.0, lit[-1] / max(w - 1, 1))


def health_color(pct: float):
    return (0, int(255 * pct), int(255 * (1.0 - pct)))


# ── Deadzone masking ──────────────────────────────────────────────────────────
def apply_deadzones(frame):
    h, w = frame.shape[:2]
    for fx1, fy1, fx2, fy2 in DEADZONES:
        x1, y1 = int(fx1 * w), int(fy1 * h)
        x2, y2 = int(fx2 * w), int(fy2 * h)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 0, 0), -1)
        cv2.rectangle(frame, (x1, y1), (x2, y2), (60, 60, 60), 1)
        cv2.putText(frame, "DEADZONE", (x1 + 4, y1 + 16),
                    cv2.FONT_HERSHEY_PLAIN, 1.0, (80, 80, 80), 1, cv2.LINE_AA)


# ── FPS / timing overlay ──────────────────────────────────────────────────────
def draw_perf(frame, fps, timings, device):
    lines = [
        f"{fps:4.1f} FPS",
        f"dev:  {device}",
        f"cap:  {timings['cap']*1000:4.0f} ms",
        f"inf:  {timings['inf']*1000:4.0f} ms",
        f"draw: {timings['draw']*1000:4.0f} ms",
    ]
    fw = frame.shape[1]
    y = 24
    for i, line in enumerate(lines):
        (tw, th), _ = cv2.getTextSize(line, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        x = fw - tw - 12
        cv2.rectangle(frame, (x - 6, y - th - 6), (fw - 2, y + 6), (0, 0, 0), -1)
        col = (0, 255, 0)   if (i == 0 and fps >= 25) else \
              (0, 200, 255) if (i == 0 and fps >= 15) else \
              (0, 80, 255)  if (i == 0)               else \
              (210, 210, 210)
        cv2.putText(frame, line, (x, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, col, 1, cv2.LINE_AA)
        y += th + 12


# ── Per-class body-box scale factors ─────────────────────────────────────────
# (down_mult, side_mult) applied to BODY_EXTEND_DOWN / BODY_WIDEN_EACH
def body_scale(label: str):
    l = label.lower()
    if "minion" in l:
        return (0.5, 1.0)   # half height
    if any(k in l for k in ("turret", "tower", "inhibitor", "nexus", "structure")):
        return (3.0, 1.0)   # triple height
    if any(k in l for k in ("baron", "dragon", "drake", "herald", "epic")):
        return (2.0, 2.0)   # double height AND double width
    return (1.0, 1.0)       # champions / everything else: default


# ── Main drawing: health bar box + expanded body box + label ─────────────────
def draw_detections(frame, dets, frame_h, frame_w):
    for d in dets:
        x1, y1, x2, y2 = d["box"]          # YOLO box  = health bar position
        color           = d["color"]
        label           = d["label"]
        health_pct      = d["health"]
        hcol            = health_color(health_pct)

        down_mult, side_mult = body_scale(label)
        extend_down = int(BODY_EXTEND_DOWN * down_mult)
        widen_each  = int(BODY_WIDEN_EACH  * side_mult)

        # ── 1. Health bar fill inside the YOLO box ────────────────────────────
        bar_fill_x2 = x1 + int((x2 - x1) * health_pct)
        cv2.rectangle(frame, (x1, y1), (bar_fill_x2, y2), hcol, -1)
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 1)

        # ── 2. Expanded "body" box projected below the health bar ─────────────
        bx1 = max(0,       x1 - widen_each)
        bx2 = min(frame_w, x2 + widen_each)
        by1 = y2
        by2 = min(frame_h, y2 + extend_down)
        cv2.rectangle(frame, (bx1, by1), (bx2, by2), color, BOX_THICKNESS)

        # ── 3. Label + health % just above the health bar ─────────────────────
        txt = f"{label}  {int(health_pct * 100)}%"
        (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
        lx = x1
        ly = max(y1 - 4, th + 4)
        cv2.rectangle(frame, (lx - 1, ly - th - 2), (lx + tw + 2, ly + 2),
                      (0, 0, 0), -1)
        cv2.putText(frame, txt, (lx, ly),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, color, 1, cv2.LINE_AA)


# ── Threaded screen capture ───────────────────────────────────────────────────
class ScreenGrabber(threading.Thread):
    def __init__(self, monitor_index, downscale_w):
        super().__init__(daemon=True)
        self.monitor_index = monitor_index
        self.downscale_w   = downscale_w
        self.latest        = None
        self.lock          = threading.Lock()
        self.running       = True
        self.cap_ms        = 0.0
        self.monitor_info  = None
        self._first        = threading.Event()

    def run(self):
        sct = mss()
        monitor = sct.monitors[self.monitor_index]
        self.monitor_info = monitor
        while self.running:
            t0  = time.perf_counter()
            raw = np.asarray(sct.grab(monitor))
            frm = raw[:, :, :3]
            if self.downscale_w and frm.shape[1] > self.downscale_w:
                scale = self.downscale_w / frm.shape[1]
                frm   = cv2.resize(frm, (self.downscale_w, int(frm.shape[0] * scale)),
                                   interpolation=cv2.INTER_AREA)
            else:
                frm = np.ascontiguousarray(frm)
            with self.lock:
                self.latest = frm
                self.cap_ms = time.perf_counter() - t0
            self._first.set()
        sct.close()

    def wait_first(self, timeout=10):
        return self._first.wait(timeout)

    def get(self):
        with self.lock:
            return None if self.latest is None else self.latest.copy()

    def stop(self):
        self.running = False


# ── Device ────────────────────────────────────────────────────────────────────
def pick_device():
    if not _HAS_TORCH:
        return "cpu"
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


# ── Main ──────────────────────────────────────────────────────────────────────
def run():
    print("Loading model...")
    model  = YOLO("best.pt")
    device = pick_device()
    print(f"Device: {device.upper()}")
    try:
        model.to(device)
    except Exception as e:
        print(f"  model.to() warning: {e}")

    class_names = model.names
    if isinstance(class_names, dict):
        class_names = [class_names[k] for k in sorted(class_names)]

    print("Warming up...")
    dummy = np.zeros((INFERENCE_IMGSZ, INFERENCE_IMGSZ, 3), dtype=np.uint8)
    model.predict(source=dummy, imgsz=INFERENCE_IMGSZ, conf=CONF_THRESHOLD,
                  verbose=False, half=USE_HALF, device=device)

    tmp = mss()
    monitors = tmp.monitors
    print("Monitors:")
    for i, m in enumerate(monitors):
        print(f"  [{i}] {m['width']}x{m['height']} @ ({m['left']},{m['top']})")
    tmp.close()

    MONITOR_IDX = min(1, len(monitors) - 1)
    grabber = ScreenGrabber(MONITOR_IDX, CAPTURE_DOWNSCALE_W)
    grabber.start()
    if not grabber.wait_first(10):
        print("No frame — abort.")
        grabber.stop()
        return
    print(f"Capturing monitor [{MONITOR_IDX}]. Press 'q' to quit.")

    fps_ema   = None
    prev_t    = time.perf_counter()
    timings   = {"cap": 0.0, "inf": 0.0, "draw": 0.0}
    frame_idx = 0
    dets      = []

    while True:
        frame = grabber.get()
        if frame is None:
            continue
        apply_deadzones(frame)
        fh, fw = frame.shape[:2]
        t1 = time.perf_counter()

        # ── Inference ─────────────────────────────────────────────────────────
        frame_idx += 1
        if frame_idx % INFERENCE_EVERY_N == 0:
            results = model.predict(source=frame, imgsz=INFERENCE_IMGSZ,
                                    conf=CONF_THRESHOLD, verbose=False,
                                    half=USE_HALF, device=device)
            dets = []
            boxes = results[0].boxes
            if boxes is not None:
                for box in boxes:
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    cls_id = int(box.cls[0])
                    label  = class_names[cls_id] if class_names and cls_id < len(class_names) \
                             else f"cls{cls_id}"
                    # Health % from the YOLO crop (which IS the health bar)
                    crop   = frame[max(0,y1):min(fh,y2), max(0,x1):min(fw,x2)]
                    health = estimate_health_pct(crop)
                    dets.append({
                        "box":    (x1, y1, x2, y2),
                        "label":  label,
                        "color":  class_color(label),
                        "health": health,
                    })
        t2 = time.perf_counter()

        # ── Draw ──────────────────────────────────────────────────────────────
        out = frame.copy()
        draw_detections(out, dets, fh, fw)
        t3 = time.perf_counter()

        # ── FPS smoothing ─────────────────────────────────────────────────────
        now    = time.perf_counter()
        dt     = now - prev_t
        prev_t = now
        if dt > 0:
            inst    = 1.0 / dt
            fps_ema = inst if fps_ema is None else 0.9 * fps_ema + 0.1 * inst

        a = 0.9
        timings["cap"]  = grabber.cap_ms
        timings["inf"]  = a * timings["inf"]  + (1 - a) * (t2 - t1)
        timings["draw"] = a * timings["draw"] + (1 - a) * (t3 - t2)

        draw_perf(out, fps_ema or 0.0, timings, device)
        cv2.imshow("YOLOv8 Live", out)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    grabber.stop()
    grabber.join(timeout=1)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    run()